In [1]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [2]:
date = read_table("select * from sc_gold.dim_date")
state= read_table("select * from sc_gold.dim_state")

In [3]:
df = read_table(
    """
    select l.*, g.gdp_per_capita, g.population
    from sc_bronze.datagov_labour l
    left join sc_silver.gdp_per_capita g
    on l.date = g.date and l.state = g.state
    order by l.date, l.state
    """
)
df

,state,date,total_labour,total_emp,total_unemp,total_olf,lab_unemp_rate,employment_pop_ratio,gdp_per_capita,population
0,Johor,2016-01-01,1639100.0,1580600.0,58500.0,820700.0,3.6,62.5,31481.859631,3651800.0
1,Kedah,2016-01-01,910900.0,884300.0,26600.0,508400.0,2.9,61.9,19263.988772,2119700.0
2,Kelantan,2016-01-01,657800.0,632900.0,24900.0,454800.0,3.8,55.8,12495.682084,1796700.0
3,Melaka,2016-01-01,400800.0,397300.0,3500.0,211900.0,0.9,63.9,41775.919432,901100.0
4,Negeri Sembilan,2016-01-01,479700.0,466900.0,12800.0,262100.0,2.7,61.5,37141.060675,1099300.0
...,...,...,...,...,...,...,...,...,...,...
123,Selangor,2023-01-01,3967600.0,3858900.0,108700.0,1156800.0,2.7,75.4,54469.235891,7209700.0
124,Terengganu,2023-01-01,478700.0,462400.0,16400.0,319700.0,3.4,57.9,31559.249587,1210000.0
125,W.P. Kuala Lumpur,2023-01-01,1111700.0,1077900.0,33900.0,387200.0,3.0,71.6,123686.704891,2005700.0
126,W.P. Labuan,2023-01-01,46500.0,43400.0,3100.0,22000.0,6.8,63.8,81575.222222,99000.0


In [4]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    state[["state", "state_id"]],
    on="state",
    how="left"
)


df_final = df.drop(columns=["date", "state"])
id_cols = ["date_id", "state_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [5]:
df_final["gpl_id"] = ["GPL" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["gpl_id"] + [c for c in df_final.columns if c != "gpl_id"]]
df_final

,gpl_id,date_id,state_id,total_labour,total_emp,total_unemp,total_olf,lab_unemp_rate,employment_pop_ratio,gdp_per_capita,population
0,GPL0001,DT001,ST001,1639100.0,1580600.0,58500.0,820700.0,3.6,62.5,31481.859631,3651800.0
1,GPL0002,DT001,ST002,910900.0,884300.0,26600.0,508400.0,2.9,61.9,19263.988772,2119700.0
2,GPL0003,DT001,ST003,657800.0,632900.0,24900.0,454800.0,3.8,55.8,12495.682084,1796700.0
3,GPL0004,DT001,ST004,400800.0,397300.0,3500.0,211900.0,0.9,63.9,41775.919432,901100.0
4,GPL0005,DT001,ST005,479700.0,466900.0,12800.0,262100.0,2.7,61.5,37141.060675,1099300.0
...,...,...,...,...,...,...,...,...,...,...,...
123,GPL0124,DT029,ST012,3967600.0,3858900.0,108700.0,1156800.0,2.7,75.4,54469.235891,7209700.0
124,GPL0125,DT029,ST013,478700.0,462400.0,16400.0,319700.0,3.4,57.9,31559.249587,1210000.0
125,GPL0126,DT029,ST014,1111700.0,1077900.0,33900.0,387200.0,3.0,71.6,123686.704891,2005700.0
126,GPL0127,DT029,ST015,46500.0,43400.0,3100.0,22000.0,6.8,63.8,81575.222222,99000.0


In [6]:
write_table(df_final, "sc_gold", "fact_gcp_pop_labour")

Table sc_gold.fact_gcp_pop_labour written successfully.
